# Camada Gold — Modelagem Dimensional

## Objetivo
Construir o Star Schema (5 dims + 1 fato + 3 bridges) a partir das tabelas Silver, com chaves substitutas e validação por bloco.

**Convenções deste projeto:**
- Comentários e textos: português (PT-BR)
- Identificadores (variáveis, funções): inglês (EN)
  - Constantes: `UPPER_SNAKE_CASE`
  - Demais variáveis: `lower_snake_case`
- Colunas e nomes de tabela: português, `snake_case`
- Símbolos nos logs: ✅ ⚠️ ❌

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime

# Padronização dos nomes dos schemas e volumes
CATALOG = "movies_medallion"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"

BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

LANDING_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA_NAME}/landing"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"CATALOG: {CATALOG}")
print(f"BRONZE_SCHEMA: {BRONZE_SCHEMA}")
print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")
print(f"LANDING_PATH: {LANDING_PATH}")

CATALOG: movies_medallion
BRONZE_SCHEMA: movies_medallion.bronze
SILVER_SCHEMA: movies_medallion.silver
GOLD_SCHEMA: movies_medallion.gold
LANDING_PATH: /Volumes/movies_medallion/bronze/landing


## dim_movies
Metadados descritivos de cada filme.

- Base: `silver.tb_info_filmes`
- Sinopse nula/vazia → "Sinopse não disponível"
- SK `sk_movie_id` determinística via `row_number()` ordenado por `id_filme`

In [0]:
df_silver_movies_info = spark.table(f"{SILVER_SCHEMA}.tb_info_filmes")

display(df_silver_movies_info.limit(5))

# Lista de colunas que vêm do Silver
col_list = [
    "id_filme",
    "titulo",
    "data_lancamento",
    "ano_lancamento",
    "duracao_minutos",
    "idioma_original",
    "status_filme",
    "sinopse",
]

df_dim_movies = (
    df_silver_movies_info
    .select(col_list)
    # Sinopse nula ou vazia vira texto padronizado.
    .withColumn(
        "sinopse",
        F.when(
            F.col("sinopse").isNull() | (F.trim(F.col("sinopse")) == ""),
            "Sinopse não disponível",
        ).otherwise(F.trim(F.col("sinopse"))),
    )
    # SK determinística: row_number ordenado pela chave natural.
    .withColumn(
        "sk_movie_id",
        F.row_number().over(Window.orderBy(F.col("id_filme"))).try_cast("bigint"),
    )
    .select("sk_movie_id", *col_list)
)

display(df_dim_movies.limit(5))
df_dim_movies.printSchema()

# Validações
total = df_dim_movies.count()
total_silver = df_silver_movies_info.count()
sk_unique = df_dim_movies.select("sk_movie_id").distinct().count()
id_unique = df_dim_movies.select("id_filme").distinct().count()

print(f"Total dim_movies: {total}")
print(f"Total silver:     {total_silver}")
print(f"SKs únicas:       {sk_unique}")
print(f"IDs únicos:       {id_unique}")

# Contagem bate com o Silver
if total != total_silver:
    print(f"{FAIL} dim_movies: {total} linhas vs Silver: {total_silver}")
    raise Exception(f"dim_movies: {total} linhas enquanto a Silver tem {total_silver}.")

# sk_movie_id única
if sk_unique != total:
    print(f"{FAIL} sk_movie_id duplicada em dim_movies: {sk_unique} vs {total}")
    raise Exception(f"sk_movie_id duplicada: {sk_unique} únicas para {total} linhas.")

# id_filme única
if id_unique != total:
    print(f"{FAIL} id_filme duplicado em dim_movies: {id_unique} vs {total}")
    raise Exception(f"id_filme duplicado: {id_unique} únicos para {total} linhas.")

print(f"{OK} Contagens OK — {total} filmes únicos")

# Estrutura: nomes das colunas na ordem esperada
expected_cols = [
    "sk_movie_id",
    "id_filme",
    "titulo",
    "data_lancamento",
    "ano_lancamento",
    "duracao_minutos",
    "idioma_original",
    "status_filme",
    "sinopse",
]
assert df_dim_movies.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, "
    f"obtido {df_dim_movies.columns}"
)
print(f"{OK} Estrutura OK — 9 colunas com nomes corretos")

# Estrutura: tipos esperados por coluna
expected_types = {
    "sk_movie_id": "bigint",
    "id_filme": "string",
    "titulo": "string",
    "data_lancamento": "date",
    "ano_lancamento": "int",
    "duracao_minutos": "int",
    "idioma_original": "string",
    "status_filme": "string",
    "sinopse": "string",
}
actual_types = {f.name: f.dataType.simpleString() for f in df_dim_movies.schema.fields}
assert actual_types == expected_types, (
    f"Tipos errados: esperado {expected_types}, obtido {actual_types}",
)
print(f"{OK} Tipos OK — 9 colunas com tipos corretos")

# Nenhuma coluna 100% nula
for c in df_dim_movies.columns:
    nulos = df_dim_movies.filter(F.col(c).isNull()).count()
    if nulos == total:
        print(f"{FAIL} {c} está 100% nula em dim_movies")
        raise Exception(f"Coluna {c} ficou integralmente NULL em dim_movies.")
print(f"{OK} Nenhuma coluna 100% nula")

# Persiste
(df_dim_movies
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_SCHEMA}.dim_movies"))

print(f"Tabela gravada: {GOLD_SCHEMA}.dim_movies")

# Validação da tabela gravada
df_check = spark.table(f"{GOLD_SCHEMA}.dim_movies")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(5))

id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
1000004,Purple Beatz,Purple Beatz,2022-07-07,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.,2022
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null,2020
1000007,Kyle Brownrigg: Introducing Lyle,Kyle Brownrigg: Introducing Lyle,2022-05-27,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null,2022
1000011,Worth Your Weight In Gold,O Teu Peso Em Ouro,2022-07-14,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null,2022
1000014,On Va Manquer !,On va manquer !,2018-05-15,0,fr,Lançado,null,null,2018


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
2,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
3,1000007,Kyle Brownrigg: Introducing Lyle,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
4,1000011,Worth Your Weight In Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
5,1000014,On Va Manquer !,2018-05-15,2018,0,fr,Lançado,Sinopse não disponível


root
 |-- sk_movie_id: long (nullable = false)
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)

Total dim_movies: 97879
Total silver:     97879
SKs únicas:       97879
IDs únicos:       97879
✅ Contagens OK — 97879 filmes únicos
✅ Estrutura OK — 9 colunas com nomes corretos
✅ Tipos OK — 9 colunas com tipos corretos
✅ Nenhuma coluna 100% nula
Tabela gravada: movies_medallion.gold.dim_movies
Linhas na tabela: 97879
root
 |-- sk_movie_id: long (nullable = true)
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_origina

sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
2,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
3,1000007,Kyle Brownrigg: Introducing Lyle,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
4,1000011,Worth Your Weight In Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
5,1000014,On Va Manquer !,2018-05-15,2018,0,fr,Lançado,Sinopse não disponível


## dim_genres
Catálogo único e deduplicado de gêneros cinematográficos.

- Base: `silver.tb_generos` (apenas a coluna `generos`)
- Deduplica antes de gerar SK
- SK `sk_genre_id` via `row_number()` ordenado por nome do gênero

In [0]:
df_silver_genres_info = spark.table(f"{SILVER_SCHEMA}.tb_generos")

display(df_silver_genres_info.limit(5))

df_dim_genres = (
    df_silver_genres_info
    .select("generos")
    .distinct()
    # SK determinística: row_number ordenado por nome do gênero.
    .withColumn(
        "sk_genre_id",
        F.row_number().over(Window.orderBy(F.col("generos"))).try_cast("bigint"),
    )
    .select("sk_genre_id", F.col("generos").alias("nome_genero"))
)

display(df_dim_genres.orderBy("sk_genre_id"))
df_dim_genres.printSchema()

# Validações
total = df_dim_genres.count()
total_silver = df_silver_genres_info.select("generos").distinct().count()
sk_unique = df_dim_genres.select("sk_genre_id").distinct().count()
nome_unique = df_dim_genres.select("nome_genero").distinct().count()

print(f"Total dim_genres: {total}")
print(f"Total silver (gêneros únicos): {total_silver}")
print(f"SKs únicas: {sk_unique}")
print(f"Gêneros únicos: {nome_unique}")

# Contagem bate com o Silver
if total != total_silver:
    print(f"{FAIL} dim_genres: {total} linhas vs Silver: {total_silver}")
    raise Exception(f"dim_genres: {total} linhas enquanto a Silver tem {total_silver}.")

# sk_genre_id única
if sk_unique != total:
    print(f"{FAIL} sk_genre_id duplicada em dim_genres: {sk_unique} vs {total}")
    raise Exception(f"sk_genre_id duplicada: {sk_unique} únicas para {total} linhas.")

# nome_genero única
if nome_unique != total:
    print(f"{FAIL} nome_genero duplicado em dim_genres: {nome_unique} vs {total}")
    raise Exception(f"nome_genero duplicado: {nome_unique} únicos para {total} linhas.")

print(f"{OK} Contagens OK — {total} gêneros únicos")

# Estrutura: nomes das colunas na ordem esperada
expected_cols = ["sk_genre_id", "nome_genero"]
assert df_dim_genres.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, "
    f"obtido {df_dim_genres.columns}"
)
print(f"{OK} Estrutura OK — 2 colunas com nomes corretos")

# Estrutura: tipos esperados por coluna
expected_types = {
    "sk_genre_id": "bigint",
    "nome_genero": "string",
}
actual_types = {f.name: f.dataType.simpleString() for f in df_dim_genres.schema.fields}
assert actual_types == expected_types, (
    f"Tipos errados: esperado {expected_types}, obtido {actual_types}"
)
print(f"{OK} Tipos OK — 2 colunas com tipos corretos")

# Nenhuma coluna 100% nula
for c in df_dim_genres.columns:
    nulos = df_dim_genres.filter(F.col(c).isNull()).count()
    if nulos == total:
        print(f"{FAIL} {c} está 100% nula em dim_genres")
        raise Exception(f"Coluna {c} ficou integralmente NULL em dim_genres.")
print(f"{OK} Nenhuma coluna 100% nula")

# Persiste
(df_dim_genres
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_SCHEMA}.dim_genres"))

print(f"Tabela gravada: {GOLD_SCHEMA}.dim_genres")

# Validação da tabela gravada
df_check = spark.table(f"{GOLD_SCHEMA}.dim_genres")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.orderBy("sk_genre_id"))


id_filme,generos
329865,Mystery
269149,Animation
429617,Science Fiction
330459,Action
127380,Family


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_genre_id,nome_genero
1,Action
2,Adventure
3,Animation
4,Comedy
5,Crime
6,Documentary
7,Drama
8,Family
9,Fantasy
10,History


root
 |-- sk_genre_id: long (nullable = false)
 |-- nome_genero: string (nullable = true)

Total dim_genres: 19
Total silver (gêneros únicos): 19
SKs únicas: 19
Gêneros únicos: 19
✅ Contagens OK — 19 gêneros únicos
✅ Estrutura OK — 2 colunas com nomes corretos
✅ Tipos OK — 2 colunas com tipos corretos
✅ Nenhuma coluna 100% nula
Tabela gravada: movies_medallion.gold.dim_genres
Linhas na tabela: 19
root
 |-- sk_genre_id: long (nullable = true)
 |-- nome_genero: string (nullable = true)



sk_genre_id,nome_genero
1,Action
2,Adventure
3,Animation
4,Comedy
5,Crime
6,Documentary
7,Drama
8,Family
9,Fantasy
10,History


## gold.dim_people
Pessoas físicas envolvidas nas obras (1 linha por combinação nome+tipo).
- Filtra apenas Ator, Diretor, Roteirista (Produtora vai pra dim_companies)
- Remove column shift: números puros (0.0, 1.013) e nomes longos (> 6 palavras)
- Distinto por (nome, tipo) — uma pessoa pode ter mais de um papel
- SK gerada com row_number ordenado por (nome, tipo)

In [0]:
df_silver_people_info = spark.table(f"{SILVER_SCHEMA}.tb_pessoas_empresas")

display(df_silver_people_info.limit(5))
df_silver_people_info.printSchema()


def filter_people(df):
    return (
        # Filtra para pessoas (exclui produtora)
        df.filter(F.col("tipo").isin("Ator", "Diretor", "Roteirista"))
        # Sem produtoras na tabela, dígito no nome provavelmente é lixo 
        .filter(~F.col("nome").rlike(r"^\d+(\.\d+)?$"))
        .filter(F.size(F.split(F.col("nome"), r"\s+")) <= 6)
    )

df_dim_people = (
    filter_people(df_silver_people_info)
    .select("nome", "tipo")
    .distinct()
    # SK determinística: row_number ordenado por (nome, tipo).
    .withColumn(
        "sk_person_id",
        F.row_number()
        .over(Window.orderBy(F.col("nome"), F.col("tipo")))
        .try_cast("bigint"),
    )
    .select(
        "sk_person_id",
        F.col("nome").alias("nome_pessoa"),
        F.col("tipo").alias("tipo_pessoa"),
    )
)

display(df_dim_people.orderBy("sk_person_id").limit(10))
df_dim_people.printSchema()


# Validações
total = df_dim_people.count()
total_silver = filter_people(df_silver_people_info).select("nome", "tipo").distinct().count()
sk_unique = df_dim_people.select("sk_person_id").distinct().count()
nome_tipo_unique = df_dim_people.select("nome_pessoa", "tipo_pessoa").distinct().count()

print(f"Total dim_people: {total}")
print(f"Total silver (nome+tipo únicos): {total_silver}")
print(f"SKs únicas: {sk_unique}")
print(f"Pares (nome, tipo) únicos: {nome_tipo_unique}")

# Verifica se a contagem bate com o Silver filtrado
if total != total_silver:
    print(f"{FAIL} dim_people: {total} linhas vs Silver: {total_silver}")
    raise Exception(f"dim_people: {total} linhas enquanto a Silver tem {total_silver}.")

# Verifica se a sk_person_id única
if sk_unique != total:
    print(f"{FAIL} sk_person_id duplicada: {sk_unique} vs {total}")
    raise Exception(f"sk_person_id duplicada: {sk_unique} únicas para {total} linhas.")

# Verifica se (nome, tipo) é único
if nome_tipo_unique != total:
    print(f"{FAIL} (nome, tipo) duplicado: {nome_tipo_unique} vs {total}")
    raise Exception(f"(nome, tipo) duplicado: {nome_tipo_unique} únicos para {total} linhas.")

print(f"{OK} Contagens OK — {total} pessoas únicas")

# Estrutura: nomes das colunas
expected_cols = ["sk_person_id", "nome_pessoa", "tipo_pessoa"]
assert df_dim_people.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, obtido {df_dim_people.columns}"
)
print(f"{OK} Estrutura OK — 3 colunas com nomes corretos")

# Estrutura: tipos
expected_types = {
    "sk_person_id": "bigint",
    "nome_pessoa": "string",
    "tipo_pessoa": "string",
}
actual_types = {f.name: f.dataType.simpleString() for f in df_dim_people.schema.fields}
assert actual_types == expected_types, (
    f"Tipos errados: esperado {expected_types}, obtido {actual_types}"
)
print(f"{OK} Tipos OK — 3 colunas com tipos corretos")

# Verifica os tipos de pessoa: só Ator, Diretor, Roteirista
tipos_validos = {"Ator", "Diretor", "Roteirista"}
tipos_encontrados = {
    row["tipo_pessoa"]
    for row in df_dim_people.select("tipo_pessoa").distinct().collect()
}
if tipos_encontrados != tipos_validos:
    print(f"{FAIL} Tipos inesperados: {tipos_encontrados - tipos_validos}")
    raise Exception(f"Tipos inesperados: {tipos_encontrados - tipos_validos}")
print(f"{OK} Tipos de pessoa OK")

# Verifica se nenhuma coluna é 100% nula
for c in df_dim_people.columns:
    nulos = df_dim_people.filter(F.col(c).isNull()).count()
    if nulos == total:
        print(f"{FAIL} {c} está 100% nula")
        raise Exception(f"Coluna {c} ficou integralmente NULL em dim_people.")
print(f"{OK} Nenhuma coluna 100% nula")

# Persiste
(df_dim_people
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_SCHEMA}.dim_people"))

print(f"Tabela gravada: {GOLD_SCHEMA}.dim_people")

# Validação da tabela gravada
df_check = spark.table(f"{GOLD_SCHEMA}.dim_people")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.orderBy("sk_person_id").limit(10))

id_filme,nome,tipo
299536,Don Cheadle,Ator
263115,Hugh Jackman,Ator
354912,Anthony Gonzalez,Ator
419430,Daniel Kaluuya,Ator
424694,Mike Myers,Ator


root
 |-- id_filme: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- tipo: string (nullable = true)



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_person_id,nome_pessoa,tipo_pessoa
1,'ana Ika,Ator
2,'e-gotti' Eric Johnson,Ator
3,'jeeva' Ravi,Ator
4,'meesai' Mohan,Ator
5,'meesai' Rajendran,Ator
6,'om' Rakesh Chaturvedi,Ator
7,'poo' Ram,Ator
8,'sunday Jeff' Silverman,Ator
9,'weird Al' Yankovic,Roteirista
10,'wáats'asdíyei Joe Yates,Diretor


root
 |-- sk_person_id: long (nullable = false)
 |-- nome_pessoa: string (nullable = true)
 |-- tipo_pessoa: string (nullable = true)

Total dim_people: 411659
Total silver (nome+tipo únicos): 411659
SKs únicas: 411659
Pares (nome, tipo) únicos: 411659
✅ Contagens OK — 411659 pessoas únicas
✅ Estrutura OK — 3 colunas com nomes corretos
✅ Tipos OK — 3 colunas com tipos corretos
✅ Tipos de pessoa OK
✅ Nenhuma coluna 100% nula
Tabela gravada: movies_medallion.gold.dim_people
Linhas na tabela: 411659
root
 |-- sk_person_id: long (nullable = true)
 |-- nome_pessoa: string (nullable = true)
 |-- tipo_pessoa: string (nullable = true)



sk_person_id,nome_pessoa,tipo_pessoa
1,'ana Ika,Ator
2,'e-gotti' Eric Johnson,Ator
3,'jeeva' Ravi,Ator
4,'meesai' Mohan,Ator
5,'meesai' Rajendran,Ator
6,'om' Rakesh Chaturvedi,Ator
7,'poo' Ram,Ator
8,'sunday Jeff' Silverman,Ator
9,'weird Al' Yankovic,Roteirista
10,'wáats'asdíyei Joe Yates,Diretor


## gold.dim_companies
Catálogo único de produtoras/estúdios.
- Filtra apenas `tipo == "Produtora"` do Silver
- SK via `row_number()` ordenado por nome
- Sem filtro extra de column shift: ganho era 0,66% (290 de 43.725 linhas) e arriscava estúdios legítimos com dígito (A24, 101 Studios, 100 Bares)

In [0]:
df_silver_companies_info = spark.table(f"{SILVER_SCHEMA}.tb_pessoas_empresas")

display(df_silver_companies_info.filter(F.col("tipo") == "Produtora").limit(5))
df_silver_companies_info.printSchema()

df_dim_companies = (
    df_silver_companies_info
    .filter(F.col("tipo") == "Produtora")
    .select("nome")
    .distinct()
    # SK determinística: row_number ordenado por nome.
    .withColumn(
        "sk_company_id",
        F.row_number().over(Window.orderBy(F.col("nome"))).try_cast("bigint"),
    )
    .select("sk_company_id", F.col("nome").alias("nome_produtora"))
)

display(df_dim_companies.orderBy("sk_company_id").limit(10))
df_dim_companies.printSchema()

# Validações
total = df_dim_companies.count()
total_silver = (
    df_silver_companies_info
    .filter(F.col("tipo") == "Produtora")
    .select("nome")
    .distinct()
    .count()
)
sk_unique = df_dim_companies.select("sk_company_id").distinct().count()
nome_unique = df_dim_companies.select("nome_produtora").distinct().count()

print(f"Total dim_companies: {total}")
print(f"Total silver (nomes únicos): {total_silver}")
print(f"SKs únicas: {sk_unique}")
print(f"Nomes únicos: {nome_unique}")

# Verifica se contagem bate com o Silver
if total != total_silver:
    print(f"{FAIL} dim_companies: {total} linhas vs Silver: {total_silver}")
    raise Exception(f"dim_companies: {total} linhas enquanto a Silver tem {total_silver}.")

# Verifica se sk_company_id é única
if sk_unique != total:
    print(f"{FAIL} sk_company_id duplicada: {sk_unique} vs {total}")
    raise Exception(f"sk_company_id duplicada: {sk_unique} únicas para {total} linhas.")

# Verifica se nome_produtora é único
if nome_unique != total:
    print(f"{FAIL} nome_produtora duplicado: {nome_unique} vs {total}")
    raise Exception(f"nome_produtora duplicado: {nome_unique} únicos para {total} linhas.")

print(f"{OK} Contagens OK — {total} produtoras únicas")

# Estrutura: nomes das colunas
expected_cols = ["sk_company_id", "nome_produtora"]
assert df_dim_companies.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, obtido {df_dim_companies.columns}"
)
print(f"{OK} Estrutura OK — 2 colunas com nomes corretos")

# Estrutura: tipos
expected_types = {
    "sk_company_id": "bigint",
    "nome_produtora": "string",
}
actual_types = {f.name: f.dataType.simpleString() for f in df_dim_companies.schema.fields}
assert actual_types == expected_types, (
    f"Tipos errados: esperado {expected_types}, obtido {actual_types}"
)
print(f"{OK} Tipos OK — 2 colunas com tipos corretos")

# Verifica se nenhuma coluna é 100% nula
for c in df_dim_companies.columns:
    nulos = df_dim_companies.filter(F.col(c).isNull()).count()
    if nulos == total:
        print(f"{FAIL} {c} está 100% nula")
        raise Exception(f"Coluna {c} ficou integralmente NULL em dim_companies.")
print(f"{OK} Nenhuma coluna 100% nula")

# Persiste
(df_dim_companies
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_SCHEMA}.dim_companies"))

print(f"Tabela gravada: {GOLD_SCHEMA}.dim_companies")

# Validação da tabela gravada
df_check = spark.table(f"{GOLD_SCHEMA}.dim_companies")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.orderBy("sk_company_id").limit(10))

id_filme,nome,tipo
424694,Gk Films,Produtora
424694,Regency Enterprises,Produtora
335983,Matt Tolmach Productions,Produtora
330457,Walt Disney Pictures,Produtora
337404,Gunn Films,Produtora


root
 |-- id_filme: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- tipo: string (nullable = true)



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_company_id,nome_produtora
1,& Extermination In An American City
2,& Space Productions
3,'s Wonderful Pictures
4,... Productions
5,.mille Et Une. Films.
6,0.0
7,0.0071
8,0.0143
9,0.6
10,0.608


root
 |-- sk_company_id: long (nullable = false)
 |-- nome_produtora: string (nullable = true)

Total dim_companies: 43725
Total silver (nomes únicos): 43725
SKs únicas: 43725
Nomes únicos: 43725
✅ Contagens OK — 43725 produtoras únicas
✅ Estrutura OK — 2 colunas com nomes corretos
✅ Tipos OK — 2 colunas com tipos corretos
✅ Nenhuma coluna 100% nula
Tabela gravada: movies_medallion.gold.dim_companies
Linhas na tabela: 43725
root
 |-- sk_company_id: long (nullable = true)
 |-- nome_produtora: string (nullable = true)



sk_company_id,nome_produtora
1,& Extermination In An American City
2,& Space Productions
3,'s Wonderful Pictures
4,... Productions
5,.mille Et Une. Films.
6,0.0
7,0.0071
8,0.0143
9,0.6
10,0.608


## gold.dim_reviews
Avaliações agregadas por filme (1 linha por filme avaliado).
- Agrega `tb_avaliacoes_usuarios` por `id_filme`: contagem + média (2 casas)
- Join com `dim_movies` para obter `sk_movie_id` (FK)
- Média convertida para DOUBLE, conforme spec §Entrega 1
- SK `sk_review_id` via `row_number()` ordenado por `sk_movie_id`

In [0]:
df_silver_reviews_info = spark.table(f"{SILVER_SCHEMA}.tb_avaliacoes_usuarios")
df_dim_movies = spark.table(f"{GOLD_SCHEMA}.dim_movies")

display(df_silver_reviews_info.limit(5))
df_silver_reviews_info.printSchema()

# Agrega avaliações por filme: contagem + média (2 casas).
# Cast para double porque avg() de decimal retorna decimal, e a spec quer DOUBLE.
# Inner join descarta órfãos (review de filme que não está na dim_movies).
df_dim_reviews = (
    df_silver_reviews_info
    .groupBy("id_filme")
    .agg(
        F.count("*").try_cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).try_cast("double").alias("nota_media_usuarios"),
    )
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    # SK determinística: row_number ordenado por sk_movie_id.
    .withColumn(
        "sk_review_id",
        F.row_number().over(Window.orderBy(F.col("sk_movie_id"))).try_cast("bigint"),
    )
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios",
    )
)

display(df_dim_reviews.orderBy("sk_review_id").limit(10))
df_dim_reviews.printSchema()

# Validações
total = df_dim_reviews.count()
sk_review_unique = df_dim_reviews.select("sk_review_id").distinct().count()
sk_movie_unique = df_dim_reviews.select("sk_movie_id").distinct().count()

# Esperado: filmes únicos com review que existem na dim_movies
esperado = (
    df_silver_reviews_info.select("id_filme").distinct()
    .join(df_dim_movies.select("id_filme"), on="id_filme", how="inner")
    .count()
)

# Órfãos: reviews cujo id_filme não está na dim_movies (dropados pelo inner join)
orfaos = (
    df_silver_reviews_info.select("id_filme").distinct()
    .join(df_dim_movies.select("id_filme"), on="id_filme", how="left_anti")
    .count()
)

print(f"Total dim_reviews: {total}")
print(f"Esperado (filmes com review na dim): {esperado}")
print(f"SKs únicas: {sk_review_unique}")
print(f"sk_movie_id únicos: {sk_movie_unique}")
print(f"Órfãos (id_filme sem par na dim_movies): {orfaos}")

# Verifica se contagem bate com o esperado
if total != esperado:
    print(f"{FAIL} dim_reviews: {total} linhas vs esperado {esperado}")
    raise Exception(f"dim_reviews: {total} linhas vs esperado {esperado}.")

# Verifica se sk_review_id é única
if sk_review_unique != total:
    print(f"{FAIL} sk_review_id duplicada: {sk_review_unique} vs {total}")
    raise Exception(f"sk_review_id duplicada: {sk_review_unique} únicas para {total} linhas.")

# Verifica se sk_movie_id é única (1 review agregada por filme)
if sk_movie_unique != total:
    print(f"{FAIL} sk_movie_id duplicado: {sk_movie_unique} vs {total}")
    raise Exception(f"sk_movie_id duplicado: {sk_movie_unique} únicos para {total} linhas.")

print(f"{OK} Contagens OK — {total} filmes com review")

# Estrutura: nomes das colunas
expected_cols = [
    "sk_review_id",
    "sk_movie_id",
    "qtd_avaliacoes_usuarios",
    "nota_media_usuarios",
]
assert df_dim_reviews.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, obtido {df_dim_reviews.columns}"
)
print(f"{OK} Estrutura OK — 4 colunas com nomes corretos")

# Estrutura: tipos
expected_types = {
    "sk_review_id": "bigint",
    "sk_movie_id": "bigint",
    "qtd_avaliacoes_usuarios": "int",
    "nota_media_usuarios": "double",
}
actual_types = {f.name: f.dataType.simpleString() for f in df_dim_reviews.schema.fields}
assert actual_types == expected_types, (
    f"Tipos errados: esperado {expected_types}, obtido {actual_types}"
)
print(f"{OK} Tipos OK — 4 colunas com tipos corretos")

# Verifica se nota_media_usuarios está no intervalo 0-10
fora_faixa = df_dim_reviews.filter(
    F.col("nota_media_usuarios").isNotNull()
    & ((F.col("nota_media_usuarios") < 0) | (F.col("nota_media_usuarios") > 10))
).count()
if fora_faixa > 0:
    print(f"{FAIL} {fora_faixa} médias fora de 0-10")
    raise Exception(f"{fora_faixa} médias fora de 0-10.")
print(f"{OK} Nota média dentro de 0-10")

# Verifica se nenhuma coluna é 100% nula
for c in df_dim_reviews.columns:
    nulos = df_dim_reviews.filter(F.col(c).isNull()).count()
    if nulos == total:
        print(f"{FAIL} {c} está 100% nula")
        raise Exception(f"Coluna {c} ficou integralmente NULL em dim_reviews.")
print(f"{OK} Nenhuma coluna 100% nula")

# Persiste
(df_dim_reviews
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_SCHEMA}.dim_reviews"))

print(f"Tabela gravada: {GOLD_SCHEMA}.dim_reviews")

# Validação da tabela gravada
df_check = spark.table(f"{GOLD_SCHEMA}.dim_reviews")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.orderBy("sk_review_id").limit(10))


id_filme,nome_usuario,nota_usuario,comentario_usuario
846452,Sérgio Freitas,6.10,Assisti até o final mas não me marcou.
493192,Felipe Ferreira 158,7.00,Filme interessante com boas atuações.
664037,Juliana Castro 595,3.90,"Fraco, não recomendo."
464493,Adriana Dias 257,0.40,Péssimo em todos os sentidos.
582666,Thiago Moreira,0.70,Não recomendo de jeito nenhum.


root
 |-- id_filme: string (nullable = true)
 |-- nome_usuario: string (nullable = true)
 |-- nota_usuario: decimal(18,2) (nullable = true)
 |-- comentario_usuario: string (nullable = true)



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
1,1,1,0.1
2,5,1,3.3
3,8,1,6.8
4,17,1,5.4
5,19,1,5.3
6,20,1,4.4
7,23,3,4.75
8,24,1,4.8
9,26,1,6.6
10,33,1,7.5


root
 |-- sk_review_id: long (nullable = false)
 |-- sk_movie_id: long (nullable = true)
 |-- qtd_avaliacoes_usuarios: integer (nullable = true)
 |-- nota_media_usuarios: double (nullable = true)

Total dim_reviews: 27303
Esperado (filmes com review na dim): 27303
SKs únicas: 27303
sk_movie_id únicos: 27303
Órfãos (id_filme sem par na dim_movies): 364
✅ Contagens OK — 27303 filmes com review
✅ Estrutura OK — 4 colunas com nomes corretos
✅ Tipos OK — 4 colunas com tipos corretos
✅ Nota média dentro de 0-10
✅ Nenhuma coluna 100% nula
Tabela gravada: movies_medallion.gold.dim_reviews
Linhas na tabela: 27303
root
 |-- sk_review_id: long (nullable = true)
 |-- sk_movie_id: long (nullable = true)
 |-- qtd_avaliacoes_usuarios: integer (nullable = true)
 |-- nota_media_usuarios: double (nullable = true)



sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
1,1,1,0.1
2,5,1,3.3
3,8,1,6.8
4,17,1,5.4
5,19,1,5.3
6,20,1,4.4
7,23,3,4.75
8,24,1,4.8
9,26,1,6.6
10,33,1,7.5


## gold.fact_movies_performance
Métricas financeiras e de engajamento por filme (1 linha por filme lançado).
- Base: `dim_movies` filtrada por `status_filme == "Lançado"`
- Left join com `tb_financeiro_filmes` e `tb_metricas_engajamento` (nem todo filme tem as duas)
- Cast das colunas para os tipos exigidos na spec §Entrega 1 (DECIMAL, DOUBLE, INT)
- `dim_reviews` **não** entra na fato — é dimensão periférica ligada por `sk_movie_id`

In [0]:
df_dim_movies = spark.table(f"{GOLD_SCHEMA}.dim_movies")
df_silver_financeiro = spark.table(f"{SILVER_SCHEMA}.tb_financeiro_filmes")
df_silver_metricas = spark.table(f"{SILVER_SCHEMA}.tb_metricas_engajamento")

display(df_silver_financeiro.limit(5))
display(df_silver_metricas.limit(5))

# Fato de filmes lançados. Left joins preservam filmes que não têm
# financeiro ou métrica cadastrados.
df_fact_movies_performance = (
    df_dim_movies
    .filter(F.col("status_filme") == "Lançado")
    .join(df_silver_financeiro, on="id_filme", how="left")
    .join(df_silver_metricas, on="id_filme", how="left")
    # Cast para os tipos exigidos na spec
    .withColumn("orcamento_usd", F.col("orcamento_usd").try_cast("decimal(18,2)"))
    .withColumn("receita_usd", F.col("receita_usd").try_cast("decimal(18,2)"))
    .withColumn("lucro_usd", F.col("lucro_usd").try_cast("decimal(18,2)"))
    .withColumn("orcamento_brl", F.col("orcamento_brl").try_cast("decimal(18,2)"))
    .withColumn("receita_brl", F.col("receita_brl").try_cast("decimal(18,2)"))
    .withColumn("lucro_brl", F.col("lucro_brl").try_cast("decimal(18,2)"))
    .withColumn("popularidade", F.col("popularidade").try_cast("double"))
    .withColumn("nota_media_tmdb", F.col("nota_media_tmdb").try_cast("double"))
    .withColumn("nota_media_imdb", F.col("nota_media_imdb").try_cast("double"))
    .select(
        "sk_movie_id",
        "orcamento_usd", "receita_usd", "lucro_usd",
        "orcamento_brl", "receita_brl", "lucro_brl",
        "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imdb", "qtd_votos_imdb",
    )
)

display(df_fact_movies_performance.limit(10))
df_fact_movies_performance.printSchema()

# Validações
total = df_fact_movies_performance.count()
sk_unique = df_fact_movies_performance.select("sk_movie_id").distinct().count()
lancados_dim = df_dim_movies.filter(F.col("status_filme") == "Lançado").count()

print(f"Total fact: {total}")
print(f"Filmes lançados na dim_movies: {lancados_dim}")
print(f"SKs únicas: {sk_unique}")

# Verifica se contagem bate com filmes lançados na dim_movies
if total != lancados_dim:
    print(f"{FAIL} fact: {total} linhas vs lançados na dim: {lancados_dim}")
    raise Exception(f"fact: {total} linhas vs {lancados_dim} filmes lançados.")

# Verifica se sk_movie_id é única
if sk_unique != total:
    print(f"{FAIL} sk_movie_id duplicada: {sk_unique} vs {total}")
    raise Exception(f"sk_movie_id duplicada: {sk_unique} únicas para {total} linhas.")

print(f"{OK} Contagens OK — {total} filmes lançados")

# Estrutura: nomes das colunas
expected_cols = [
    "sk_movie_id",
    "orcamento_usd", "receita_usd", "lucro_usd",
    "orcamento_brl", "receita_brl", "lucro_brl",
    "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
    "nota_media_imdb", "qtd_votos_imdb",
]
assert df_fact_movies_performance.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, "
    f"obtido {df_fact_movies_performance.columns}"
)
print(f"{OK} Estrutura OK — 12 colunas com nomes corretos")

# Estrutura: tipos
expected_types = {
    "sk_movie_id": "bigint",
    "orcamento_usd": "decimal(18,2)",
    "receita_usd": "decimal(18,2)",
    "lucro_usd": "decimal(18,2)",
    "orcamento_brl": "decimal(18,2)",
    "receita_brl": "decimal(18,2)",
    "lucro_brl": "decimal(18,2)",
    "popularidade": "double",
    "nota_media_tmdb": "double",
    "qtd_votos_tmdb": "int",
    "nota_media_imdb": "double",
    "qtd_votos_imdb": "int",
}
actual_types = {f.name: f.dataType.simpleString() for f in df_fact_movies_performance.schema.fields}
assert actual_types == expected_types, (
    f"Tipos errados: esperado {expected_types}, obtido {actual_types}"
)
print(f"{OK} Tipos OK — 12 colunas com tipos corretos")

# Verifica se nenhuma coluna é 100% nula
for c in df_fact_movies_performance.columns:
    nulos = df_fact_movies_performance.filter(F.col(c).isNull()).count()
    if nulos == total:
        print(f"{FAIL} {c} está 100% nula")
        raise Exception(f"Coluna {c} ficou integralmente NULL na fato.")
print(f"{OK} Nenhuma coluna 100% nula")

# Persiste
(df_fact_movies_performance
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_SCHEMA}.fact_movies_performance"))

print(f"Tabela gravada: {GOLD_SCHEMA}.fact_movies_performance")

# Validação da tabela gravada
df_check = spark.table(f"{GOLD_SCHEMA}.fact_movies_performance")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(10))

id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
284054,200000000.00,1349926083.00,9.9384E8,6.7080526916435995E9,1149926083.00,5.7142126916435995E9,85.18
447332,17000000.00,341000000.00,8.44764E7,1.6944972E9,324000000.00,1.6100208E9,95.01
337404,200000000.00,233503234.00,9.9384E8,1.1603242703927999E9,33503234.00,1.6648427039279985E8,14.35
512200,125000000.00,801693929.00,6.2115E8,3.9837774719867997E9,676693929.00,3.3626274719867997E9,84.41
348350,250000000.00,392952373.00,1.2423E9,1.9526589319115999E9,142952373.00,7.103589319115999E8,36.38


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000004,1.13,0.00,0,6.80,27
1000005,0.60,0.00,0,null,40
1000007,0.60,0.00,0,4.70,10
1000011,1.17,0.00,0,4.80,16
1000014,0.60,0.00,0,7.20,15


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,null,null,null,null,null,null,1.13,0.0,0,6.8,27
2,null,null,null,null,null,null,0.6,0.0,0,null,40
3,null,null,null,null,null,null,0.6,0.0,0,4.7,10
4,null,null,null,null,null,null,1.17,0.0,0,4.8,16
5,null,null,null,null,null,null,0.6,0.0,0,7.2,15
6,null,null,null,null,null,null,0.62,0.0,0,null,25
7,null,null,null,null,null,null,0.6,0.0,0,5.9,10
8,4700000.00,null,null,23355240.00,null,null,1.49,6.75,6,6.2,382
9,null,null,null,null,null,null,0.6,0.0,0,7.7,25
10,6000000.00,null,null,29815200.00,null,null,13.21,6.8,15,null,236


root
 |-- sk_movie_id: long (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)

Total fact: 96463
Filmes lançados na dim_movies: 96463
SKs únicas: 96463
✅ Contagens OK — 96463 filmes lançados
✅ Estrutura OK — 12 colunas com nomes corretos
✅ Tipos OK — 12 colunas com tipos corretos
✅ Nenhuma coluna 100% nula
Tabela gravada: movies_medallion.gold.fact_movies_performance
Linhas na tabela: 96463
root
 |-- sk_movie_id: long (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- rec

sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,null,null,null,null,null,null,1.13,0.0,0,6.8,27
2,null,null,null,null,null,null,0.6,0.0,0,null,40
3,null,null,null,null,null,null,0.6,0.0,0,4.7,10
4,null,null,null,null,null,null,1.17,0.0,0,4.8,16
5,null,null,null,null,null,null,0.6,0.0,0,7.2,15
6,null,null,null,null,null,null,0.62,0.0,0,null,25
7,null,null,null,null,null,null,0.6,0.0,0,5.9,10
8,4700000.00,null,null,23355240.00,null,null,1.49,6.75,6,6.2,382
9,null,null,null,null,null,null,0.6,0.0,0,7.7,25
10,6000000.00,null,null,29815200.00,null,null,13.21,6.8,15,null,236


## gold.bridge_movie_genre
Liga filmes a gêneros (N:N). 1 linha por par (filme, gênero).
- Fonte: `tb_generos` + `dim_genres` + `dim_movies`
- Join por `generos == nome_genero` e `id_filme == id_filme`
- Inner join descarta órfãos (filme ou gênero sem par)

In [0]:
df_silver_generos = spark.table(f"{SILVER_SCHEMA}.tb_generos")
df_dim_genres = spark.table(f"{GOLD_SCHEMA}.dim_genres")
df_dim_movies = spark.table(f"{GOLD_SCHEMA}.dim_movies")

display(df_silver_generos.limit(5))
display(df_dim_genres.limit(5))

df_bridge_movie_genre = (
    df_silver_generos
    .join(df_dim_genres, df_silver_generos.generos == df_dim_genres.nome_genero, "inner")
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

display(df_bridge_movie_genre.orderBy("sk_movie_id").limit(15))
df_bridge_movie_genre.printSchema()

# Validações
total = df_bridge_movie_genre.count()
pares_unicos = df_bridge_movie_genre.select("sk_movie_id", "sk_genre_id").distinct().count()

# Esperado: tb_generos filtrado por id_filme que existe na dim_movies
esperado = (
    df_silver_generos
    .join(df_dim_movies.select("id_filme"), on="id_filme", how="inner")
    .count()
)

# Órfãos: pares do silver cujo id_filme não está na dim_movies
orfaos = (
    df_silver_generos
    .join(df_dim_movies.select("id_filme"), on="id_filme", how="left_anti")
    .count()
)

print(f"Total bridge: {total}")
print(f"Pares únicos: {pares_unicos}")
print(f"Esperado: {esperado}")
print(f"Órfãos (id_filme sem par na dim_movies): {orfaos}")

# Verifica se contagem bate com o esperado
if total != esperado:
    print(f"{FAIL} bridge: {total} linhas vs esperado {esperado}")
    raise Exception(f"bridge: {total} linhas vs esperado {esperado}.")

# Verifica se pares (sk_movie_id, sk_genre_id) são únicos
if pares_unicos != total:
    print(f"{FAIL} pares duplicados: {pares_unicos} vs {total}")
    raise Exception(f"pares (sk_movie_id, sk_genre_id) duplicados: {pares_unicos} vs {total}.")

print(f"{OK} Contagens OK — {total} pares únicos")

# Estrutura: nomes das colunas
expected_cols = ["sk_movie_id", "sk_genre_id"]
assert df_bridge_movie_genre.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, obtido {df_bridge_movie_genre.columns}"
)
print(f"{OK} Estrutura OK — 2 colunas com nomes corretos")

# Estrutura: tipos
expected_types = {
    "sk_movie_id": "bigint",
    "sk_genre_id": "bigint",
}
actual_types = {f.name: f.dataType.simpleString() for f in df_bridge_movie_genre.schema.fields}
assert actual_types == expected_types, (
    f"Tipos errados: esperado {expected_types}, obtido {actual_types}"
)
print(f"{OK} Tipos OK — 2 colunas com tipos corretos")

# Verifica se nenhuma coluna é 100% nula
for c in df_bridge_movie_genre.columns:
    nulos = df_bridge_movie_genre.filter(F.col(c).isNull()).count()
    if nulos == total:
        print(f"{FAIL} {c} está 100% nula")
        raise Exception(f"Coluna {c} ficou integralmente NULL no bridge.")
print(f"{OK} Nenhuma coluna 100% nula")

# Persiste
(df_bridge_movie_genre
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_SCHEMA}.bridge_movie_genre"))

print(f"Tabela gravada: {GOLD_SCHEMA}.bridge_movie_genre")

# Validação da tabela gravada
df_check = spark.table(f"{GOLD_SCHEMA}.bridge_movie_genre")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.orderBy("sk_movie_id").limit(15))

id_filme,generos
329865,Mystery
269149,Animation
429617,Science Fiction
330459,Action
127380,Family


sk_genre_id,nome_genero
1,Action
2,Adventure
3,Animation
4,Comedy
5,Crime


sk_movie_id,sk_genre_id
1,12
1,7
1,14
2,4
3,4
5,4
6,6
7,6
8,7
10,4


root
 |-- sk_movie_id: long (nullable = true)
 |-- sk_genre_id: long (nullable = true)

Total bridge: 140464
Pares únicos: 140464
Esperado: 140464
Órfãos (id_filme sem par na dim_movies): 1724
✅ Contagens OK — 140464 pares únicos
✅ Estrutura OK — 2 colunas com nomes corretos
✅ Tipos OK — 2 colunas com tipos corretos
✅ Nenhuma coluna 100% nula
Tabela gravada: movies_medallion.gold.bridge_movie_genre
Linhas na tabela: 140464
root
 |-- sk_movie_id: long (nullable = true)
 |-- sk_genre_id: long (nullable = true)



sk_movie_id,sk_genre_id
1,12
1,7
1,14
2,4
3,4
5,4
6,6
7,6
8,7
10,4


## gold.bridge_movie_person
Liga filmes a pessoas físicas (N:N). 1 linha por par (filme, pessoa).
- Fonte: `tb_pessoas_empresas` filtrado por Ator/Diretor/Roteirista
- Join com `dim_people` por `(nome, tipo)` — não só por nome
- Join com `dim_movies` por `id_filme`
- Mesmo filtro do `dim_people` (função `filter_people`)

In [0]:

df_silver_people = spark.table(f"{SILVER_SCHEMA}.tb_pessoas_empresas")
df_dim_people = spark.table(f"{GOLD_SCHEMA}.dim_people")
df_dim_movies = spark.table(f"{GOLD_SCHEMA}.dim_movies")

display(df_dim_people.limit(5))

# Reusa o filtro do dim_people — se um nome foi removido da dim, ele não pode
# aparecer no bridge.
df_silver_people_filtered = filter_people(df_silver_people)

# Join por (nome, tipo) para casos que o mesmo nome aparece como Ator e Diretor, por exemplo
df_bridge_movie_person = (
    df_silver_people_filtered
    .join(
        df_dim_people,
        (df_silver_people_filtered.nome == df_dim_people.nome_pessoa)
        & (df_silver_people_filtered.tipo == df_dim_people.tipo_pessoa),
        "inner",
    )
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

display(df_bridge_movie_person.orderBy("sk_movie_id").limit(10))
df_bridge_movie_person.printSchema()

# Validações
total = df_bridge_movie_person.count()
pares_unicos = df_bridge_movie_person.select("sk_movie_id", "sk_person_id").distinct().count()

# Órfãos em dim_movies (id_filme do silver não está na dim)
orfaos_movies = (
    df_silver_people_filtered
    .join(df_dim_movies.select("id_filme"), on="id_filme", how="left_anti")
    .count()
)

# Órfãos em dim_people (nome+tipo do silver não está na dim)
orfaos_people = (
    df_silver_people_filtered
    .join(
        df_dim_people,
        (df_silver_people_filtered.nome == df_dim_people.nome_pessoa)
        & (df_silver_people_filtered.tipo == df_dim_people.tipo_pessoa),
        "left_anti",
    )
    .count()
)

print(f"Total bridge: {total}")
print(f"Pares únicos: {pares_unicos}")
print(f"Órfãos em dim_movies: {orfaos_movies}")
print(f"Órfãos em dim_people: {orfaos_people}")

# Verifica se pares (sk_movie_id, sk_person_id) são únicos
if pares_unicos != total:
    print(f"{FAIL} pares duplicados: {pares_unicos} vs {total}")
    raise Exception(f"pares (sk_movie_id, sk_person_id) duplicados: {pares_unicos} vs {total}.")

print(f"{OK} Contagens OK — {total} pares únicos")

# Estrutura: nomes das colunas
expected_cols = ["sk_movie_id", "sk_person_id"]
assert df_bridge_movie_person.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, obtido {df_bridge_movie_person.columns}"
)
print(f"{OK} Estrutura OK — 2 colunas com nomes corretos")

# Estrutura: tipos
expected_types = {
    "sk_movie_id": "bigint",
    "sk_person_id": "bigint",
}
actual_types = {f.name: f.dataType.simpleString() for f in df_bridge_movie_person.schema.fields}
assert actual_types == expected_types, (
    f"Tipos errados: esperado {expected_types}, obtido {actual_types}"
)
print(f"{OK} Tipos OK — 2 colunas com tipos corretos")

# Verifica se nenhuma coluna é 100% nula
for c in df_bridge_movie_person.columns:
    nulos = df_bridge_movie_person.filter(F.col(c).isNull()).count()
    if nulos == total:
        print(f"{FAIL} {c} está 100% nula")
        raise Exception(f"Coluna {c} ficou integralmente NULL no bridge.")
print(f"{OK} Nenhuma coluna 100% nula")

# Persiste
(df_bridge_movie_person
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_SCHEMA}.bridge_movie_person"))

print(f"Tabela gravada: {GOLD_SCHEMA}.bridge_movie_person")

# Validação da tabela gravada
df_check = spark.table(f"{GOLD_SCHEMA}.bridge_movie_person")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.orderBy("sk_movie_id").limit(10))

sk_person_id,nome_pessoa,tipo_pessoa
1,'ana Ika,Ator
2,'e-gotti' Eric Johnson,Ator
3,'jeeva' Ravi,Ator
4,'meesai' Mohan,Ator
5,'meesai' Rajendran,Ator


sk_movie_id,sk_person_id
1,117189
1,160944
1,36532
1,230935
1,373895
2,255881
2,7222
3,217604
3,255881
3,217605


root
 |-- sk_movie_id: long (nullable = true)
 |-- sk_person_id: long (nullable = true)

Total bridge: 750905
Pares únicos: 750905
Órfãos em dim_movies: 9479
Órfãos em dim_people: 0
✅ Contagens OK — 750905 pares únicos
✅ Estrutura OK — 2 colunas com nomes corretos
✅ Tipos OK — 2 colunas com tipos corretos
✅ Nenhuma coluna 100% nula
Tabela gravada: movies_medallion.gold.bridge_movie_person
Linhas na tabela: 750905
root
 |-- sk_movie_id: long (nullable = true)
 |-- sk_person_id: long (nullable = true)



sk_movie_id,sk_person_id
1,160944
1,373895
1,36532
1,117189
1,230935
2,255881
2,7222
3,217604
3,217605
3,255881


## gold.bridge_movie_company
Liga filmes a produtoras (N:N). 1 linha por par (filme, produtora).
- Fonte: `tb_pessoas_empresas` filtrado por `tipo == "Produtora"`
- Join com `dim_companies` por `nome == nome_produtora`
- Join com `dim_movies` por `id_filme`

In [0]:

df_silver_company = spark.table(f"{SILVER_SCHEMA}.tb_pessoas_empresas")
df_dim_companies = spark.table(f"{GOLD_SCHEMA}.dim_companies")
df_dim_movies = spark.table(f"{GOLD_SCHEMA}.dim_movies")

display(df_dim_companies.limit(5))

# Mesmo filtro do dim_companies: sem filtro extra de column shift
df_silver_company_filtered = df_silver_company.filter(F.col("tipo") == "Produtora")

# Join apenas por nome, uma unica opcão
df_bridge_movie_company = (
    df_silver_company_filtered
    .join(
        df_dim_companies,
        df_silver_company_filtered.nome == df_dim_companies.nome_produtora,
        "inner",
    )
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

display(df_bridge_movie_company.orderBy("sk_movie_id").limit(10))
df_bridge_movie_company.printSchema()

# Validações
total = df_bridge_movie_company.count()
pares_unicos = df_bridge_movie_company.select("sk_movie_id", "sk_company_id").distinct().count()

# Órfãos em dim_movies (id_filme do silver não está na dim)
orfaos_movies = (
    df_silver_company_filtered
    .join(df_dim_movies.select("id_filme"), on="id_filme", how="left_anti")
    .count()
)

# Órfãos em dim_companies (nome do silver não está na dim)
orfaos_companies = (
    df_silver_company_filtered
    .join(df_dim_companies, df_silver_company_filtered.nome == df_dim_companies.nome_produtora, "left_anti")
    .count()
)

print(f"Total bridge: {total}")
print(f"Pares únicos: {pares_unicos}")
print(f"Órfãos em dim_movies: {orfaos_movies}")
print(f"Órfãos em dim_companies: {orfaos_companies}")

# Verifica se pares (sk_movie_id, sk_company_id) são únicos
if pares_unicos != total:
    print(f"{FAIL} pares duplicados: {pares_unicos} vs {total}")
    raise Exception(f"pares (sk_movie_id, sk_company_id) duplicados: {pares_unicos} vs {total}.")

print(f"{OK} Contagens OK — {total} pares únicos")

# Estrutura: nomes das colunas
expected_cols = ["sk_movie_id", "sk_company_id"]
assert df_bridge_movie_company.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, obtido {df_bridge_movie_company.columns}"
)
print(f"{OK} Estrutura OK — 2 colunas com nomes corretos")

# Estrutura: tipos
expected_types = {
    "sk_movie_id": "bigint",
    "sk_company_id": "bigint",
}
actual_types = {f.name: f.dataType.simpleString() for f in df_bridge_movie_company.schema.fields}
assert actual_types == expected_types, (
    f"Tipos errados: esperado {expected_types}, obtido {actual_types}"
)
print(f"{OK} Tipos OK — 2 colunas com tipos corretos")

# Verifica se nenhuma coluna é 100% nula
for c in df_bridge_movie_company.columns:
    nulos = df_bridge_movie_company.filter(F.col(c).isNull()).count()
    if nulos == total:
        print(f"{FAIL} {c} está 100% nula")
        raise Exception(f"Coluna {c} ficou integralmente NULL no bridge.")
print(f"{OK} Nenhuma coluna 100% nula")

# Persiste
(df_bridge_movie_company
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_SCHEMA}.bridge_movie_company"))

print(f"Tabela gravada: {GOLD_SCHEMA}.bridge_movie_company")

# Validação da tabela gravada
df_check = spark.table(f"{GOLD_SCHEMA}.bridge_movie_company")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.orderBy("sk_movie_id").limit(10))

sk_company_id,nome_produtora
1,& Extermination In An American City
2,& Space Productions
3,'s Wonderful Pictures
4,... Productions
5,.mille Et Une. Films.


sk_movie_id,sk_company_id
2,20108
4,27991
5,15213
5,27339
6,8670
6,33940
8,22042
8,7717
8,14909
10,33634


root
 |-- sk_movie_id: long (nullable = true)
 |-- sk_company_id: long (nullable = true)

Total bridge: 114337
Pares únicos: 114337
Órfãos em dim_movies: 1297
Órfãos em dim_companies: 0
✅ Contagens OK — 114337 pares únicos
✅ Estrutura OK — 2 colunas com nomes corretos
✅ Tipos OK — 2 colunas com tipos corretos
✅ Nenhuma coluna 100% nula
Tabela gravada: movies_medallion.gold.bridge_movie_company
Linhas na tabela: 114337
root
 |-- sk_movie_id: long (nullable = true)
 |-- sk_company_id: long (nullable = true)



sk_movie_id,sk_company_id
2,20108
4,27991
5,15213
5,27339
6,8670
6,33940
8,22042
8,7717
8,14909
10,33634


## Resumo Final — Camada Gold
Apenas reporta. Validações por bloco já deram raise antes.

In [0]:
# Resumo final: apenas reporta. Validações por bloco já deram raise antes.

tables = [
    "dim_movies",
    "dim_genres",
    "dim_people",
    "dim_companies",
    "dim_reviews",
    "fact_movies_performance",
    "bridge_movie_genre",
    "bridge_movie_person",
    "bridge_movie_company",
]

print("=" * 60)
print("RESUMO DA CAMADA GOLD")
print("=" * 60)

print(f"Tabelas em {GOLD_SCHEMA}:\n")
display(spark.sql(f"SHOW TABLES IN {GOLD_SCHEMA}"))

for t in tables:
    full = f"{GOLD_SCHEMA}.{t}"
    n = spark.table(full).count()
    print(f"{OK} {full}: {n} linhas")

print(f"Executado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

RESUMO DA CAMADA GOLD
Tabelas em movies_medallion.gold:



database,tableName,isTemporary
gold,bridge_movie_company,false
gold,bridge_movie_genre,false
gold,bridge_movie_person,false
gold,dim_companies,false
gold,dim_genres,false
gold,dim_movies,false
gold,dim_people,false
gold,dim_reviews,false
gold,fact_movies_performance,false


✅ movies_medallion.gold.dim_movies: 97879 linhas
✅ movies_medallion.gold.dim_genres: 19 linhas
✅ movies_medallion.gold.dim_people: 411659 linhas
✅ movies_medallion.gold.dim_companies: 43725 linhas
✅ movies_medallion.gold.dim_reviews: 27303 linhas
✅ movies_medallion.gold.fact_movies_performance: 96463 linhas
✅ movies_medallion.gold.bridge_movie_genre: 140464 linhas
✅ movies_medallion.gold.bridge_movie_person: 750905 linhas
✅ movies_medallion.gold.bridge_movie_company: 114337 linhas
🕐 Executado em: 2026-10-16 18:02:28
